# 01 — Data Exploration

Explore the A-share market data loaded via Qlib:
- Initialize Qlib with CN data
- List CSI300 constituent stocks
- Plot price trends for sample stocks
- Show volume distribution
- Data quality overview

In [ ]:
import sys
from pathlib import Path

# Setup path so notebooks can import big_a from src/
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"] = 120

print(f"Project root: {PROJECT_ROOT}")
print(f"Src dir: {SRC_DIR}")

## 1. Initialize Qlib

In [ ]:
from big_a.qlib_config import init_qlib

init_qlib()

## 2. CSI300 Constituent Stocks

In [ ]:
from qlib.data import D

instruments = D.instruments("csi300")
stock_list = list(instruments)
print(f"CSI300 stock count: {len(stock_list)}")
print(f"Sample: {stock_list[:10]}")

## 3. Price Trends for Sample Stocks

In [ ]:
sample_stocks = stock_list[:5]
fields = ["$close"]

price_data = D.features(
    sample_stocks,
    fields=fields,
    start_time="2024-01-01",
    end_time="2024-12-31",
)

fig, ax = plt.subplots(figsize=(14, 6))
for stock in sample_stocks:
    stock_prices = price_data.loc[stock]["$close"]
    normalized = stock_prices / stock_prices.iloc[0]
    ax.plot(normalized.index, normalized.values, label=stock, linewidth=1)

ax.set_title("Normalized Close Prices (CSI300 Sample, 2024)")
ax.set_ylabel("Normalized Price")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Volume Distribution

In [ ]:
vol_data = D.features(
    sample_stocks,
    fields=["$volume"],
    start_time="2024-01-01",
    end_time="2024-12-31",
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Volume over time
ax = axes[0]
for stock in sample_stocks:
    vol = vol_data.loc[stock]["$volume"]
    ax.plot(vol.index, vol.values, label=stock, linewidth=0.8, alpha=0.7)
ax.set_title("Daily Trading Volume")
ax.set_ylabel("Volume")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# Volume histogram
ax = axes[1]
all_volumes = vol_data["$volume"].dropna()
ax.hist(np.log10(all_volumes + 1), bins=50, alpha=0.7, edgecolor="black")
ax.set_title("Log10(Volume) Distribution")
ax.set_xlabel("log10(Volume + 1)")
ax.set_ylabel("Frequency")

plt.tight_layout()
plt.show()

## 5. Data Quality Overview

In [ ]:
from big_a.data.validation import generate_data_report

report = generate_data_report(
    market="csi300",
    start_date="2024-01-01",
    end_date="2024-12-31",
)

print("=== Calendar Integrity ===")
print(f"  Valid: {report['calendar']['valid']}")
print(f"  Total trading days: {report['calendar']['total_days']}")

print("\n=== Stock Coverage ===")
print(f"  Stock count: {report['stock_coverage']['stock_count']}")
print(f"  Valid (~300): {report['stock_coverage']['valid']}")

print("\n=== NaN Ratios ===")
for field, ratio in report['nan_ratio'].items():
    print(f"  {field}: {ratio:.4f}")

print("\n=== Price Continuity ===")
print(f"  Valid (no >10% gaps): {report['price_continuity']['valid']}")
print(f"  Anomaly count: {len(report['price_continuity']['anomalies'])}")